# 12 — 実C++コードを端から端まで追う

## 背景
ここまでのNotebookは式を小さなPythonへ写した。この章では上流commit
`a7f381c036` の実コードを、閉ループの呼出順に戻して読む。

## 目的
各blockについて、入力、C++シンボル、数式、出力、次の呼出先を一行で説明できるようにする。

## 結論
制御の中心は `LeggedController::update()` の短い配線である。複雑さは各block内部にあり、
500 Hzの配線は **推定 → policyの現在値 → WeightedWbc → hybrid joint** の順を崩さない。

## 実行境界
このcall graphは `external/legged_control/` のcommit `a7f381c0367e98e31c01336e678eef47e304d40d` にある
**ROS1 + OCS2原実装**を説明する。`src/legged_control_mujoco/adapter.py` はproject所有の
ROS-free実行境界であり、gait/state/input/WBC/hybrid-commandの契約を移植する一方、
OCS2 SQPを瞬時force plannerとMuJoCo acceleration-level WBCへ置換する。
したがってadapter結果を「OCS2 SQPを実行した結果」と呼ばない。

主な正本:
- `external/legged_control/legged_controllers/src/LeggedController.cpp`
- `external/legged_control/legged_controllers/src/TargetTrajectoriesPublisher.cpp`
- `external/legged_control/legged_estimation/src/LinearKalmanFilter.cpp`
- `external/legged_control/legged_interface/src/LeggedInterface.cpp`
- `external/legged_control/legged_wbc/src/WbcBase.cpp`
- `external/legged_control/legged_wbc/src/WeightedWbc.cpp`


## 検証範囲に関する必須注記

このprojectでは **ROS2 portを作成・compile・実行していない**。したがってROS2 parityは
**NOT VERIFIED / FAIL-CLOSED** である。上流commit `a7f381c0367e98e31c01336e678eef47e304d40d` はROS1実装であり、
project所有MuJoCo adapterはOCS2のhorizon SQPを瞬時force plannerへ、
Pinocchio/qpOASES WBCをMuJoCo acceleration inverse dynamicsへ置換し、
元のestimator/hardware経路も持たない。保存済み30 scenario dataが示すのはadapter挙動だけで、
上流 `legged_control` の性能でもROS2移行の検証でもない。


In [1]:
# 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、`from pathlib import Path` の依存を明示して再現可能な実行環境を作る。
from pathlib import Path
# 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、`import numpy as np` の依存を明示して再現可能な実行環境を作る。
import numpy as np
# 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、`import matplotlib.pyplot as plt` の依存を明示して再現可能な実行環境を作る。
import matplotlib.pyplot as plt

# 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、`ROOT` を後続計算で使う明示的な中間量として設定する。 数式: `ROOT = Path.cwd()` の演算・変換をPythonで評価する。
ROOT = Path.cwd()
# 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、`for candidate in [ROOT, *ROOT.parents]:` の反復範囲を固定して各sampleを処理する。 数式: `for candidate in [ROOT, *ROOT.parents]:` の演算・変換をPythonで評価する。
for candidate in [ROOT, *ROOT.parents]:
    # 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、`if (candidate / "pyproject.toml").exists():` の条件で安全側の実行分岐を選ぶ。 数式: `if (candidate / "pyproject.toml").exists():` の演算・変換をPythonで評価する。
    if (candidate / "pyproject.toml").exists():
        # 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、`ROOT` を後続計算で使う明示的な中間量として設定する。 数式: `ROOT = candidate` の演算・変換をPythonで評価する。
        ROOT = candidate
        # 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、`break` をこの章の処理順に沿って実行する。
        break

# 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、`np.set_printoptions(precision` を後続計算で使う明示的な中間量として設定する。 数式: `np.set_printoptions(precision=4, suppress=True)` の演算・変換をPythonで評価する。
np.set_printoptions(precision=4, suppress=True)
# 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、直前の式・構造へ `plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True})` の要素または終端を対応付ける。
plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True})
# 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、`print("repository:", ROOT)` の観測値を表示して判定根拠を残す。
print("repository:", ROOT)


repository: /home/takuya/work/mpc_dog


## 実行時call graph

```text
LeggedController::init
  ├─ setupLeggedInterface
  │    └─ LeggedInterface::setupOptimalControlProblem
  │         ├─ dynamics: LeggedRobotDynamicsAD
  │         ├─ cost: LeggedRobotQuadraticTrackingCost
  │         └─ constraints: zero force/velocity, friction, collision
  ├─ setupMpc
  │    ├─ SqpMpc
  │    ├─ GaitReceiver
  │    └─ RosReferenceManager
  ├─ setupMrt
  │    └─ thread: advanceMpc() @ 100 Hz
  ├─ setupStateEstimate
  │    └─ KalmanFilterEstimate
  └─ WeightedWbc

LeggedHWLoop / gazebo_ros_control @ 500 Hz
  └─ LeggedController::update
       ├─ updateStateEstimation
       ├─ setCurrentObservation
       ├─ updatePolicy + evaluatePolicy(now)
       ├─ WeightedWbc::update
       ├─ SafetyChecker::check
       └─ HybridJointHandle::setCommand
```


In [2]:
# --- Block 1: 上流の500 Hz配線を、コメント付きでそのまま読む ---
# この文字列は実行用Pythonではなく、照合commitのC++抜粋を教材として表示する。
# 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、`cpp_update` を後続計算で使う明示的な中間量として設定する。 数式: `cpp_update = r"""` の演算・変換をPythonで評価する。
cpp_update = r"""
void LeggedController::update(time, period) {
  updateStateEstimation(time, period);
  // IMU・q・dq・接地 → rbdState(36) → centroidal observation x(24)

  mpcMrtInterface_->setCurrentObservation(currentObservation_);
  // 現在x(24)を100 Hz NMPC threadへ共有する

  mpcMrtInterface_->updatePolicy();
  // NMPC threadが完成させた最新policyを500 Hz側へ取り込む

  evaluatePolicy(now, x, optimizedState, optimizedInput, plannedMode);
  // 未来列全部ではなく「今」の x*(24), u*(24), mode だけを切り出す

  vector_t z = wbc_->update(xStar, uStar, rbdState, mode, period);
  vector_t torque = z.tail(12);
  // WBC決定変数 z=[qdd(18), Fc(12), tau(12)] の末尾だけを使う

  setCommand(qStar, dqStar, 0, 3, torque);
  // tau_cmd = tau + 0*(q*-q) + 3*(dq*-dq)
}
"""
# 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、`print(cpp_update)` の観測値を表示して判定根拠を残す。
print(cpp_update)



void LeggedController::update(time, period) {
  updateStateEstimation(time, period);
  // IMU・q・dq・接地 → rbdState(36) → centroidal observation x(24)

  mpcMrtInterface_->setCurrentObservation(currentObservation_);
  // 現在x(24)を100 Hz NMPC threadへ共有する

  mpcMrtInterface_->updatePolicy();
  // NMPC threadが完成させた最新policyを500 Hz側へ取り込む

  evaluatePolicy(now, x, optimizedState, optimizedInput, plannedMode);
  // 未来列全部ではなく「今」の x*(24), u*(24), mode だけを切り出す

  vector_t z = wbc_->update(xStar, uStar, rbdState, mode, period);
  vector_t torque = z.tail(12);
  // WBC決定変数 z=[qdd(18), Fc(12), tau(12)] の末尾だけを使う

  setCommand(qStar, dqStar, 0, 3, torque);
  // tau_cmd = tau + 0*(q*-q) + 3*(dq*-dq)
}



In [3]:
# --- Block 2: 同じ配線をshape付きPython契約にする ---
# 意図: C++の型に隠れた次元をassertし、block間の誤接続を検出する。
# 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、`controller_update_contract` の責務を独立関数として定義する。
def controller_update_contract(rbd_state, observation_x, policy_x, policy_u, wbc_solution):
    # 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、`assert rbd_state.shape == (36,) # KF / rigid-body state` を不変条件として即時検査する。 数式: `assert rbd_state.shape == (36,) # KF / rigid-body state` の演算・変換をPythonで評価する。
    assert rbd_state.shape == (36,)       # KF / rigid-body state
    # 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、`assert observation_x.shape == (24,) # NMPC initial state` を不変条件として即時検査する。 数式: `assert observation_x.shape == (24,) # NMPC initial state` の演算・変換をPythonで評価する。
    assert observation_x.shape == (24,)   # NMPC initial state
    # 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、`assert policy_x.shape == (24,) # optimized state at now` を不変条件として即時検査する。 数式: `assert policy_x.shape == (24,) # optimized state at now` の演算・変換をPythonで評価する。
    assert policy_x.shape == (24,)        # optimized state at now
    # 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、`assert policy_u.shape == (24,) # GRF12 + joint velocity12 at now` を不変条件として即時検査する。 数式: `assert policy_u.shape == (24,) # GRF12 + joint velocity12 at now` の演算・変換をPythonで評価する。
    assert policy_u.shape == (24,)        # GRF12 + joint velocity12 at now
    # 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、`assert wbc_solution.shape == (42,) # qdd18 + force12 + torque12` を不変条件として即時検査する。 数式: `assert wbc_solution.shape == (42,) # qdd18 + force12 + torque12` の演算・変換をPythonで評価する。
    assert wbc_solution.shape == (42,)    # qdd18 + force12 + torque12

    # 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、`q_des` を後続計算で使う明示的な中間量として設定する。 数式: `q_des = policy_x[12:24] # centroidal stateのjoint angle block` の演算・変換をPythonで評価する。
    q_des = policy_x[12:24]               # centroidal stateのjoint angle block
    # 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、`dq_des` を後続計算で使う明示的な中間量として設定する。 数式: `dq_des = policy_u[12:24] # NMPC input後半はjoint velocity` の演算・変換をPythonで評価する。
    dq_des = policy_u[12:24]              # NMPC input後半はjoint velocity
    # 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、`tau_ff` を後続計算で使う明示的な中間量として設定する。 数式: `tau_ff = wbc_solution[30:42] # WBC末尾12だけがmotor feedforward` の演算・変換をPythonで評価する。
    tau_ff = wbc_solution[30:42]           # WBC末尾12だけがmotor feedforward
    # 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、`hybrid` を後続計算で使う明示的な中間量として設定する。 数式: `hybrid = np.c_[q_des, dq_des,` の演算・変換をPythonで評価する。
    hybrid = np.c_[q_des, dq_des,
                   # 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、`np.zeros(12),           # Kp` を後続計算で使う明示的な中間量として設定する。 数式: `np.zeros(12), # Kp=0` の演算・変換をPythonで評価する。
                   np.zeros(12),           # Kp=0
                   # 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、`np.full(12, 3.0),       # Kd` を後続計算で使う明示的な中間量として設定する。 数式: `np.full(12, 3.0), # Kd=3` の演算・変換をPythonで評価する。
                   np.full(12, 3.0),       # Kd=3
                   # 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、直前の式・構造へ `tau_ff]` の要素または終端を対応付ける。
                   tau_ff]
    # 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、`return hybrid # 12 joints × [q*,dq*,Kp,Kd,ff]` の値を次の制御境界へ返す。 数式: `return hybrid # 12 joints × [q*,dq*,Kp,Kd,ff]` の演算・変換をPythonで評価する。
    return hybrid                          # 12 joints × [q*,dq*,Kp,Kd,ff]

# 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、`hybrid` を後続計算で使う明示的な中間量として設定する。 数式: `hybrid = controller_update_contract(` の演算・変換をPythonで評価する。
hybrid = controller_update_contract(
    # 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、直前の式・構造へ `np.zeros(36), np.zeros(24), np.zeros(24), np.zeros(24), np.zeros(42)` の要素または終端を対応付ける。 数式: `np.zeros(36), np.zeros(24), np.zeros(24), np.zeros(24), np.zeros(42)` の演算・変換をPythonで評価する。
    np.zeros(36), np.zeros(24), np.zeros(24), np.zeros(24), np.zeros(42)
# 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、直前の式・構造へ `)` の要素または終端を対応付ける。
)
# 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、`print("hybrid command shape:", hybrid.shape)` の観測値を表示して判定根拠を残す。
print("hybrid command shape:", hybrid.shape)


hybrid command shape: (12, 5)


## Block 3 — 参照生成: C++と式

`cmdVelToTargetTrajectories()` は
```cpp
cmdVelRot = getRotationMatrixFromZyxEulerAngles(zyx) * cmdVel.head(3);
target.x = current.x + cmdVelRot.x * TIME_TO_TARGET;
target.y = current.y + cmdVelRot.y * TIME_TO_TARGET;
target.yaw = current.yaw + cmdVel(3) * TIME_TO_TARGET;
trajectories.stateTrajectory[0].head(3) = cmdVelRot;
trajectories.stateTrajectory[1].head(3) = cmdVelRot;
```
を行う。対応式は
\[
v_W=R_{ZYX}v_{cmd},\quad p_{xy}^+=p_{xy}+v_{W,xy}T,\quad
\psi^+=\psi+\dot\psi T.
\]
出力は2時刻、状態2×24、入力2×24。入力trajectoryは次元合わせのzeroで、
tracking cost側はweight-compensating inputを使う。


## Block 4 — 推定: C++と式

`KalmanFilterEstimate` constructorは
`numState = 6 + 3*numContacts = 18`,
`numObserve = 2*3*numContacts + numContacts = 28` を作る。
`update()` は
\[
A_{p,v}=\Delta tI,\quad B_p=\frac12\Delta t^2I,\quad B_v=\Delta tI
\]
を毎周期更新する。`StateEstimateBase`から姿勢と関節を受け、
KFでbase位置・速度と足world位置を更新する。

```text
q,dq ─→ Pinocchio FK ─→ 足のbase相対位置/速度 ─┐
IMU orientation,a ─→ world acceleration ───────┼→ KF xHat(18)
contact ─→ Q/Rを接地/遊脚で切替 ───────────────┘
```


In [4]:
# --- Block 5: WBCの実C++行列をshapeと式へ戻す ---
# C++:
#   a << data.M, -j_.transpose(), -s.transpose();
#   b = -data.nle;
# 数式:
#   [M, -J^T, -S^T] [qdd,Fc,tau]^T = -nle
# 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、`nq, nf, ntau` を後続計算で使う明示的な中間量として設定する。 数式: `nq, nf, ntau = 18, 12, 12` の演算・変換をPythonで評価する。
nq, nf, ntau = 18, 12, 12
# 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、`M` を後続計算で使う明示的な中間量として設定する。 数式: `M = np.eye(nq)` の演算・変換をPythonで評価する。
M = np.eye(nq)
# 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、`J` を後続計算で使う明示的な中間量として設定する。 数式: `J = np.zeros((nf, nq))` の演算・変換をPythonで評価する。
J = np.zeros((nf, nq))
# 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、`S` を後続計算で使う明示的な中間量として設定する。 数式: `S = np.c_[np.zeros((ntau, 6)), np.eye(ntau)]` の演算・変換をPythonで評価する。
S = np.c_[np.zeros((ntau, 6)), np.eye(ntau)]
# 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、`A_eom` を後続計算で使う明示的な中間量として設定する。 数式: `A_eom = np.c_[M, -J.T, -S.T]` の演算・変換をPythonで評価する。
A_eom = np.c_[M, -J.T, -S.T]
# 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、`assert A_eom.shape == (18, 42)` を不変条件として即時検査する。 数式: `assert A_eom.shape == (18, 42)` の演算・変換をPythonで評価する。
assert A_eom.shape == (18, 42)

# C++ frictionPyramic:
# [ 0, 0,-1]F <= 0       -> Fz >= 0
# [±1, 0,-mu]F <= 0      -> |Fx| <= mu Fz
# [ 0,±1,-mu]F <= 0      -> |Fy| <= mu Fz
# 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、`mu` を後続計算で使う明示的な中間量として設定する。 数式: `mu = 0.3` の演算・変換をPythonで評価する。
mu = 0.3
# 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、`D_friction` を後続計算で使う明示的な中間量として設定する。 数式: `D_friction = np.array([` の演算・変換をPythonで評価する。
D_friction = np.array([
    # 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、直前の式・構造へ `[0,0,-1], [1,0,-mu], [-1,0,-mu], [0,1,-mu], [0,-1,-mu]` の要素または終端を対応付ける。 数式: `[0,0,-1], [1,0,-mu], [-1,0,-mu], [0,1,-mu], [0,-1,-mu]` の演算・変換をPythonで評価する。
    [0,0,-1], [1,0,-mu], [-1,0,-mu], [0,1,-mu], [0,-1,-mu]
# 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、直前の式・構造へ `])` の要素または終端を対応付ける。
])
# 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、`test_force` を後続計算で使う明示的な中間量として設定する。 数式: `test_force = np.array([5.0, 3.0, 30.0])` の演算・変換をPythonで評価する。
test_force = np.array([5.0, 3.0, 30.0])
# 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、`print("EoM A shape:", A_eom.shape)` の観測値を表示して判定根拠を残す。
print("EoM A shape:", A_eom.shape)
# 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、`print("friction inequalities D F:", D_friction @ test_force, "<= 0 is fe…` の観測値を表示して判定根拠を残す。 数式: `print("friction inequalities D F:", D_friction @ test_force, "<= 0 is fe…` の演算・変換をPythonで評価する。
print("friction inequalities D F:", D_friction @ test_force, "<= 0 is feasible")
# 背景: 上流ROS1 C++のblock境界を実コードから追う。目的: 呼出順・shape・数式の対応を検査するため、`assert np.all(D_friction @ test_force <= 0)` を不変条件として即時検査する。 数式: `assert np.all(D_friction @ test_force <= 0)` の演算・変換をPythonで評価する。
assert np.all(D_friction @ test_force <= 0)


EoM A shape: (18, 42)
friction inequalities D F: [-30.  -4. -14.  -6. -12.] <= 0 is feasible


## Block 6 — WeightedWbcの解法

実コードはhard constraintsを
`formulateFloatingBaseEomTask + torqueLimits + frictionCone + noContactMotion`
で作り、soft taskを
`swingLeg*100 + baseAccel*1 + contactForce*0.01`
で作る。

\[
H=A_{soft}^TA_{soft},\qquad g=-A_{soft}^Tb_{soft}
\]
をqpOASESへ渡し、`nWsr=20`で解く。solver return codeを分岐せず
`getPrimalSolution`するため、failure fallbackは実装改善候補である。

## 全体理解の確認
1. `optimizedInput[:12]` と `[12:]` は何か。
2. 推定modeと`plannedMode`のどちらがWBC接触flagになるか。
3. frictionがNMPCではsoft円錐、WBCではhard pyramidである影響は何か。
4. `HierarchicalWbc`がincludeされても既定で動かない根拠はどこか。
5. `setCommand(...,0,3,tau)` の位置項が消えることを式で示せるか。

次のNotebookは教育用proxyであり、この上流契約そのものの性能試験ではない。
